In [7]:
import numpy as np
import pandas as pd

#importation fichier .parquet
httrain= pd.read_parquet("history_toulouse_train.parquet").copy()
httest=pd.read_parquet("history_toulouse_test.parquet").copy()
httest

,horizon,bikes_+15min,bikes_+30min,bikes_+45min,bikes_+60min,bikes_+75min,bikes_+90min,bikes_+105min,bikes_+120min
station,commit_at,,,,,,,,
00001 - POIDS DE L'HUILE,2026-07-31 21:45:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
00002 - LAFAYETTE,2026-07-31 23:45:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
00003 - ALSACE - LORRAINE - POMME,2025-11-10 11:45:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
00003 - ALSACE-LORRAINE - POMME,2026-07-31 21:45:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
00003 - POMME,2025-01-05 05:45:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
NARBONNE - CLOTASSES,2025-08-22 19:15:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
PASTEUR - LAVOISIER,2025-08-05 10:15:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
PLACE GEORGES BRASSENS,2025-08-05 11:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
#ajout serie(colonne) heure en fomrat datetime
httrain["jour"]=pd.to_datetime(httrain.index.get_level_values("commit_at")).day_name()
httrain["heure"]=pd.to_datetime(httrain.index.get_level_values("commit_at")).hour
httrain["minute"]=pd.to_datetime(httrain.index.get_level_values("commit_at")).minute


httrain

bikes  stands      jour  heure  \
station                  commit_at                                             
00001 - POIDS DE L'HUILE 2023-08-05 12:15:00     10       9  Saturday     12   
                         2023-08-05 12:30:00     10       9  Saturday     12   
                         2023-08-05 12:45:00     10       9  Saturday     12   
                         2023-08-05 13:00:00     11       8  Saturday     13   
                         2023-08-05 13:15:00     11       8  Saturday     13   
...                                             ...     ...       ...    ...   
TEST BEE                 2026-02-12 15:00:00      0       3  Thursday     15   
                         2026-02-12 15:15:00      0       3  Thursday     15   
                         2026-02-12 15:30:00      0       3  Thursday     15   
                         2026-02-12 15:45:00      0       3  Thursday     15   
                         2026-02-12 16:00:00      0       3  Thursday     16   

                                              minute  
station                  commit_at                    
00001 - POIDS DE L'HUILE 2023-08-05 12:15:00      15  
                         2023-08-05 12:30:00      30  
                         2023-08-05 12:45:00      45  
                         2023-08-05 13:00:00       0  
                         2023-08-05 13:15:00      15  
...                                              ...  
TEST BEE                 2026-02-12 15:00:00       0  
                         2026-02-12 15:15:00      15  
                         2026-02-12 15:30:00      30  
                         2026-02-12 15:45:00      45  
                         2026-02-12 16:00:00       0  

[40012622 rows x 5 columns]

In [26]:
#calcul moyenne des velo par quart d'heure
meanbh = httrain.groupby(["station","jour", "heure", "minute"])["bikes"].mean()
     
     
meanbh


station                   jour       heure  minute
00001 - POIDS DE L'HUILE  Friday     0      0         1.987179
                                            15        1.955128
                                            30        1.743590
                                            45        1.756410
                                     1      0         1.647436
                                                        ...   
TEST BEE                  Wednesday  22     45        1.000000
                                     23     0         1.000000
                                            15        1.000000
                                            30        1.000000
                                            45        1.000000
Name: bikes, Length: 532897, dtype: float64

In [30]:
# ajout des resultat dans httest
target_columns = [
    "bikes_+15min", "bikes_+30min", "bikes_+45min", "bikes_+60min",
    "bikes_+75min", "bikes_+90min", "bikes_+105min", "bikes_+120min"
]

delta_hour = [15, 30, 45, 60, 75, 90, 105, 120]


forecast= pd.to_datetime(httest.index.get_level_values("commit_at"))
stations= httest.index.get_level_values(0)

for columns in range(8):
    
    time_delta= forecast + pd.Timedelta(minutes=delta_hour[columns])
    index_recherche= pd.MultiIndex.from_arrays([
        stations,
        time_delta.day_name().astype(str),
        time_delta.hour.astype(int),
        time_delta.minute.astype(int)
    ], names=["station", "jour" , "heure", "minute"])


    
    moyennes_futures = cles_recherche.map(meanbh)
    

    httest[target_columns[columns]] =  httest[target_columns[columns]].fillna(moyennes_futures) #.round().astype(int)


httest

station                            commit_at          
00001 - POIDS DE L'HUILE           2026-07-31 21:45:00           (00001 - POIDS DE L'HUILE, Friday, 23, 45)
00002 - LAFAYETTE                  2026-07-31 23:45:00                 (00002 - LAFAYETTE, Saturday, 1, 45)
00003 - ALSACE - LORRAINE - POMME  2025-11-10 11:45:00    (00003 - ALSACE - LORRAINE - POMME, Monday, 13...
00003 - ALSACE-LORRAINE - POMME    2026-07-31 21:45:00    (00003 - ALSACE-LORRAINE - POMME, Friday, 23, 45)
00003 - POMME                      2025-01-05 05:45:00                       (00003 - POMME, Sunday, 7, 45)
                                                                                ...                        
NARBONNE - CLOTASSES               2025-08-22 19:15:00               (NARBONNE - CLOTASSES, Friday, 21, 15)
PASTEUR - LAVOISIER                2025-08-05 10:15:00               (PASTEUR - LAVOISIER, Tuesday, 12, 15)
PLACE GEORGES BRASSENS             2025-08-05 11:00:00             (PLACE GEORGES

,horizon,bikes_+15min,bikes_+30min,bikes_+45min,bikes_+60min,bikes_+75min,bikes_+90min,bikes_+105min,bikes_+120min
station,commit_at,,,,,,,,
00001 - POIDS DE L'HUILE,2026-07-31 21:45:00,5.554839,5.206452,4.722581,4.432258,4.135484,3.625806,3.193548,2.941935
00002 - LAFAYETTE,2026-07-31 23:45:00,5.690323,5.580645,5.322581,5.180645,4.838710,4.774194,4.477419,4.316129
00003 - ALSACE - LORRAINE - POMME,2025-11-10 11:45:00,11.094340,10.320755,10.962264,10.584906,9.471698,9.094340,8.018868,7.981132
00003 - ALSACE-LORRAINE - POMME,2026-07-31 21:45:00,6.000000,6.027027,5.027027,4.540541,3.621622,3.486486,2.972973,1.837838
00003 - POMME,2025-01-05 05:45:00,7.175676,7.297297,7.364865,7.418919,7.500000,7.459459,7.378378,7.648649
...,...,...,...,...,...,...,...,...,...
NARBONNE - CLOTASSES,2025-08-22 19:15:00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
PASTEUR - LAVOISIER,2025-08-05 10:15:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
PLACE GEORGES BRASSENS,2025-08-05 11:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [11]:
httest.to_parquet(path= "prevision/prediction_fct1.parquet", engine='auto', index=False) 

In [13]:
hour_forecast= (pd.to_datetime(httest.index.get_level_values("commit_at"))).copy().hour
print(hour_forecast)
minute_forecast= (pd.to_datetime(httest.index.get_level_values("commit_at"))).copy().minute
print(minute_forecast)

Index([21, 23, 11, 21,  5,  7, 11, 22,  5,  5,
       ...
       10, 10, 18, 21, 20, 19, 10, 11, 12, 16],
      dtype='int32', name='commit_at', length=800)
Index([45, 45, 45, 45, 45, 15,  0, 30, 45, 45,
       ...
        0,  0, 30, 15, 15, 15, 15,  0, 15,  0],
      dtype='int32', name='commit_at', length=800)
